# 02 - ACM Title Checks

In this notebook, I check the paper sample against ACM front matter PDFs.

The cached ACM front matter PDFs used for this check are in `(../../assets/validation_pdfs/)`.

I use two checks here.

1. I check whether the papers in my OpenAlex/DBLP sample appear in the
   corresponding ACM front matter PDF.
2. When the ACM table of contents can be parsed cleanly, I also check the
   reverse direction: whether ACM lists papers that are missing from my sample.

## 1 - Setup

In [1]:
import difflib
import re
import shutil
import subprocess
import unicodedata

import pandas as pd

from pathlib import Path

try:
    import pypdf
except ImportError:
    import PyPDF2 as pypdf


In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_2_data_dir = project_folder / "step_2_data"
step_2_artifacts_dir = project_folder / "step_2_artifacts"
prepared_dir = step_2_data_dir / "prepared"
intermediate_dir = step_2_data_dir / "intermediate" / "paper_title_checks"
summary_tables_dir = step_2_artifacts_dir / "summary_tables"
dependency_tables_dir = step_2_artifacts_dir / "dependency_tables"
check_tables_dir = step_2_artifacts_dir / "check_tables"
pdf_dir = project_folder / "assets" / "validation_pdfs"

intermediate_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)

print(project_folder)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2 - Load Paper Data

In [3]:
papers = pd.read_parquet(prepared_dir / "all_papers" / "all_papers_filtered.parquet")
papers = papers.sort_values(["conference", "conference_year", "issue", "doi"]).reset_index(drop=True)

print(papers.shape)
display(
    papers
    .groupby(["conference", "conference_year", "issue"])
    .size()
    .reset_index(name="n_papers")
)


(2528, 13)


,conference,conference_year,issue,n_papers
0,ICFP,2017,ICFP,44
1,ICFP,2018,ICFP,40
2,ICFP,2019,ICFP,39
3,ICFP,2020,ICFP,36
4,ICFP,2021,ICFP,35
5,ICFP,2022,ICFP,35
6,ICFP,2023,ICFP,33
7,ICFP,2024,ICFP,35
8,ICFP,2025,ICFP,36
9,OOPSLA,2017,OOPSLA,66


## 3 - Text Normalization

In [4]:
def fix_pypdf_artifacts(text):
    text = (
        text.replace("ﬀ", "ff")
        .replace("ﬁ", "fi")
        .replace("ﬂ", "fl")
        .replace("ﬃ", "ffi")
        .replace("ﬄ", "ffl")
    )
    text = re.sub(r"/([a-z])_([a-z])", r"\1\2", text, flags=re.IGNORECASE)
    return text


def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = fix_pypdf_artifacts(text).lower()
    text = re.sub(r"([a-z])-\s+([a-z])", r"\1\2", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    text = text.translate(str.maketrans({
        "⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",
        "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9",
    }))
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def pdf_text(pdf_path):
    reader = pypdf.PdfReader(str(pdf_path))
    text = []

    for page in reader.pages:
        text.append(page.extract_text() or "")

    return normalize_text("\n".join(text))


def title_in_pdf(title, pdf_text_norm, window=3):
    words = normalize_text(title).split()

    if not words:
        return False
    if len(words) <= window:
        return " ".join(words) in pdf_text_norm

    for i in range(len(words) - window + 1):
        fragment = " ".join(words[i:i + window])
        if fragment in pdf_text_norm:
            return True

    return False


## 4 - PDF Lookup

In [5]:
def pdf_paths_for_cell(conference, year):
    conf = conference.lower()

    if conference in {"OOPSLA1", "OOPSLA2"}:
        path = pdf_dir / "oopsla" / f"oopsla_{year}_{conf}_acm_frontmatter_2026-05-08.pdf"
        return [path] if path.exists() else []

    path = pdf_dir / conf / f"{conf}_{year}_acm_frontmatter_2026-05-08.pdf"
    return [path] if path.exists() else []


pdf_inventory = []

for conference, year in (
    papers[["conference", "conference_year"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
):
    paths = pdf_paths_for_cell(conference, int(year))
    pdf_inventory.append({
        "conference": conference,
        "conference_year": int(year),
        "pdf_count": len(paths),
        "pdf_paths": "; ".join(str(path.relative_to(project_folder)) for path in paths),
    })

pdf_inventory = pd.DataFrame(pdf_inventory)

display(pdf_inventory)


,conference,conference_year,pdf_count,pdf_paths
0,ICFP,2017,1,assets/validation_pdfs/icfp/icfp_2017_acm_fron...
1,ICFP,2018,1,assets/validation_pdfs/icfp/icfp_2018_acm_fron...
2,ICFP,2019,1,assets/validation_pdfs/icfp/icfp_2019_acm_fron...
3,ICFP,2020,1,assets/validation_pdfs/icfp/icfp_2020_acm_fron...
4,ICFP,2021,1,assets/validation_pdfs/icfp/icfp_2021_acm_fron...
5,ICFP,2022,1,assets/validation_pdfs/icfp/icfp_2022_acm_fron...
6,ICFP,2023,1,assets/validation_pdfs/icfp/icfp_2023_acm_fron...
7,ICFP,2024,1,assets/validation_pdfs/icfp/icfp_2024_acm_fron...
8,ICFP,2025,1,assets/validation_pdfs/icfp/icfp_2025_acm_fron...
9,OOPSLA,2017,1,assets/validation_pdfs/oopsla/oopsla_2017_acm_...


## 5 - Data to PDF Title Check

In [6]:
pdf_text_cache = {}
rows = []

for cell in pdf_inventory.itertuples(index=False):
    conference = cell.conference
    year = int(cell.conference_year)
    paths = pdf_paths_for_cell(conference, year)

    if paths:
        text_norm = " ".join(
            pdf_text_cache.setdefault(path, pdf_text(path))
            for path in paths
        )
    else:
        text_norm = ""

    cell_papers = papers[
        (papers["conference"] == conference)
        & (papers["conference_year"] == year)
    ]

    for paper in cell_papers.itertuples(index=False):
        rows.append({
            "conference": conference,
            "conference_year": year,
            "issue": paper.issue,
            "work_id": paper.work_id,
            "doi": paper.doi,
            "title": paper.title,
            "type": paper.type,
            "source": getattr(paper, "source", None),
            "pdf_count": len(paths),
            "in_pdf": title_in_pdf(paper.title, text_norm) if paths else False,
        })

title_check = pd.DataFrame(rows)

summary = (
    title_check
    .groupby(["conference", "conference_year"], as_index=False)
    .agg(
        n_papers=("doi", "size"),
        n_in_pdf=("in_pdf", "sum"),
        pdf_count=("pdf_count", "max"),
    )
)
summary["n_missing"] = summary["n_papers"] - summary["n_in_pdf"]
summary = summary.sort_values(["conference", "conference_year"]).reset_index(drop=True)

display(summary)


,conference,conference_year,n_papers,n_in_pdf,pdf_count,n_missing
0,ICFP,2017,44,44,1,0
1,ICFP,2018,40,40,1,0
2,ICFP,2019,39,39,1,0
3,ICFP,2020,36,36,1,0
4,ICFP,2021,35,35,1,0
5,ICFP,2022,35,35,1,0
6,ICFP,2023,33,33,1,0
7,ICFP,2024,35,35,1,0
8,ICFP,2025,36,36,1,0
9,OOPSLA,2017,66,66,1,0


## 6 - Inspect Data Titles Missing from PDFs

In [7]:
missing = (
    title_check[~title_check["in_pdf"]]
    .sort_values(["conference", "conference_year", "title"])
    .reset_index(drop=True)
)

print("Missing titles:", len(missing))
if len(missing):
    display(missing[[
        "conference", "conference_year", "issue", "doi", "title", "type", "source", "pdf_count",
    ]])


Missing titles: 0


## 7 - PDF to OpenAlex Matching Table

This table is the main validation output. Each row starts from one title listed
in an ACM front matter PDF and then records the matched OpenAlex row, when one
is found. I keep OOPSLA1 and OOPSLA2 separate.


In [ ]:
def pdf_layout_text(pdf_path):
    if shutil.which("pdftotext"):
        completed = subprocess.run(
            ["pdftotext", "-layout", str(pdf_path), "-"],
            check=True,
            capture_output=True,
            text=True,
        )
        return completed.stdout

    reader = pypdf.PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def pdf_year_from_path(pdf_path):
    match = re.search(r"_(20\d{2})_", pdf_path.name)
    return int(match.group(1)) if match else None


def pdf_issue_from_path(pdf_path, conference):
    name = pdf_path.name.lower()

    if "oopsla1" in name:
        return "OOPSLA1"
    if "oopsla2" in name:
        return "OOPSLA2"

    return conference


def pdf_volume_from_path(pdf_path, conference, year):
    header = pdf_layout_text(pdf_path).splitlines()[:5]
    header = " ".join(header)
    match = re.search(r"Volume\s+(\d+)", header)

    if match:
        return int(match.group(1))

    if conference in {"ICFP", "POPL", "OOPSLA", "OOPSLA1", "OOPSLA2"}:
        return year - 2016
    if conference == "PLDI" and year >= 2023:
        return year - 2016

    return pd.NA


def content_lines_from_pdf(pdf_path):
    text = pdf_layout_text(pdf_path)
    lines = text.splitlines()

    starts = [i for i, line in enumerate(lines) if line.strip() == "Contents"]
    if not starts:
        return []

    start = starts[0] + 1
    end_candidates = [
        i for i, line in enumerate(lines[start:], start)
        if line.strip().startswith("Author Index")
    ]
    end = end_candidates[0] if end_candidates else len(lines)

    return lines[start:end]


def line_ends_with_arabic_page(line):
    text = line.rstrip()

    if re.search(r"\.{3,}.*\b\d{1,5}\s*$", text):
        return True

    # Some ACM front-matter entries end with a plain page number, without dot
    # leaders. Short lines such as section headings are not paper entries.
    return len(text) >= 65 and re.search(r"\s\d{1,5}\s*$", text) is not None


def line_ends_with_roman_page(line):
    return re.search(r"\.{3,}.*\b([ivxlcdm]+)\s*$", line.strip(), flags=re.IGNORECASE) is not None


def pdf_page_from_block(block):
    match = re.search(r"(\d{1,5})\s*$", block.strip())
    return int(match.group(1)) if match else pd.NA


def content_blocks_from_pdf(pdf_path):
    blocks = []
    block = []
    skip_lines = {
        "Frontmatter",
        "Papers",
        "Regular Papers",
        "Research Papers",
        "Author Index",
    }

    for raw_line in content_lines_from_pdf(pdf_path):
        line = raw_line.replace("\x0c", "").rstrip()
        if not line.strip():
            continue

        stripped = line.strip()
        if stripped in skip_lines:
            block = []
            continue

        if line_ends_with_roman_page(line):
            block = []
            continue

        block.append(line)
        if line_ends_with_arabic_page(line):
            block_text = "\n".join(block)
            blocks.append(block_text)
            block = []

    return blocks


def title_from_pdf_block(block):
    title_lines = []

    for raw_line in block.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if "—" in line or " - " in line:
            break
        if title_lines and (re.search(r"\.{3,}", line) or line_ends_with_arabic_page(line)):
            break

        line = re.sub(r"\.{3,}.*$", "", line).strip()
        line = re.sub(r"\s+\d{2,5}\s*$", "", line).strip()
        if line:
            title_lines.append(line)

    title = ""
    for line in title_lines:
        if title.endswith("-"):
            title = title[:-1] + line
        else:
            title = (title + " " + line).strip()

    return title


def pdf_title_candidates(block):
    lines = [line.strip() for line in block.splitlines() if line.strip()]
    before_author = []

    for line in lines:
        if "—" in line or " - " in line:
            break
        before_author.append(line)

    if not before_author:
        before_author = lines[:1]

    candidates = []
    max_drop = min(3, max(0, len(before_author) - 1))

    for drop in range(max_drop + 1):
        candidate = title_from_pdf_block("\n".join(before_author[drop:]))
        if candidate:
            candidates.append(candidate)

    seen = set()
    unique = []
    for candidate in candidates:
        key = normalize_text(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)

    return unique


def title_match_score(left, right):
    left_norm = normalize_text(left)
    right_norm = normalize_text(right)

    if not left_norm or not right_norm:
        return 0.0
    if left_norm == right_norm:
        return 1.0
    if left_norm in right_norm or right_norm in left_norm:
        return 0.97

    return difflib.SequenceMatcher(None, left_norm, right_norm).ratio()


def trim_pdf_title_with_openalex(candidate, openalex_title):
    raw_tokens = re.split(r"\s+", candidate.strip())
    openalex_tokens = normalize_text(openalex_title).split()
    candidate_tokens = []
    token_owners = []

    for i, token in enumerate(raw_tokens):
        pieces = normalize_text(token).split()
        candidate_tokens.extend(pieces)
        token_owners.extend([i] * len(pieces))

    if not candidate_tokens or not openalex_tokens:
        return candidate

    n = len(openalex_tokens)
    for start in range(0, len(candidate_tokens) - n + 1):
        if candidate_tokens[start:start + n] == openalex_tokens:
            raw_start = token_owners[start]
            raw_end = token_owners[start + n - 1]
            return " ".join(raw_tokens[raw_start:raw_end + 1])

    best_span = candidate
    best_score = title_match_score(candidate, openalex_title)
    min_len = max(3, n - 4)
    max_len = min(len(raw_tokens), n + 6)

    for start in range(len(raw_tokens)):
        for end in range(start + min_len, min(len(raw_tokens), start + max_len) + 1):
            span = " ".join(raw_tokens[start:end])
            score = title_match_score(span, openalex_title)
            if score > best_score:
                best_score = score
                best_span = span

    if best_score >= 0.86:
        return best_span

    return candidate


def best_openalex_match(block, cell_papers):
    candidates = pdf_title_candidates(block)
    best_score = 0.0
    best_pdf_title = candidates[0] if candidates else title_from_pdf_block(block)
    best_row = None

    for candidate in candidates:
        for row in cell_papers.itertuples(index=False):
            score = title_match_score(candidate, row.title)
            candidate_len = len(normalize_text(candidate))
            best_len = len(normalize_text(best_pdf_title))
            if score > best_score or (
                abs(score - best_score) < 0.001 and candidate_len < best_len
            ):
                best_score = score
                best_pdf_title = candidate
                best_row = row

    if best_score < 0.86:
        block_norm = normalize_text(block)

        for row in cell_papers.itertuples(index=False):
            if title_in_pdf(row.title, block_norm, window=4):
                best_score = 0.97
                best_pdf_title = best_pdf_title if best_pdf_title else row.title
                best_row = row

    if best_score < 0.86:
        best_row = None
    else:
        best_pdf_title = trim_pdf_title_with_openalex(best_pdf_title, best_row.title)
        best_score = title_match_score(best_pdf_title, best_row.title)

    return best_pdf_title, best_row, best_score


def skip_unmatched_pdf_block(pdf_title, block):
    title = normalize_text(pdf_title)

    if not title:
        return True
    if "—" in block or " - " in block:
        return False

    return len(title.split()) <= 3


def comparable(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if text.lower() in {"nan", "none", ""}:
        return pd.NA

    if re.fullmatch(r"\d+(\.0)?", text):
        return int(float(text))

    return text


def values_match(left, right):
    left = comparable(left)
    right = comparable(right)

    if pd.isna(left) and pd.isna(right):
        return True
    if pd.isna(left) or pd.isna(right):
        return False

    return left == right


validation_rows = []

for cell in pdf_inventory.itertuples(index=False):
    conference = cell.conference
    year = int(cell.conference_year)
    paths = pdf_paths_for_cell(conference, year)
    matched_work_ids_for_cell = set()

    cell_papers = papers[
        (papers["conference"] == conference)
        & (papers["conference_year"] == year)
    ]

    for path in paths:
        pdf_year = pdf_year_from_path(path)
        pdf_issue = pdf_issue_from_path(path, conference)
        pdf_volume = pdf_volume_from_path(path, conference, year)
        blocks = content_blocks_from_pdf(path)

        for order, block in enumerate(blocks, start=1):
            pdf_title, matched, score = best_openalex_match(block, cell_papers)

            if matched is None:
                if skip_unmatched_pdf_block(pdf_title, block):
                    continue

                openalex_title = pd.NA
                openalex_work_id = pd.NA
                openalex_doi = pd.NA
                openalex_year = pd.NA
                openalex_volume = pd.NA
                openalex_issue = pd.NA
                openalex_source = pd.NA
                title_match = False
                year_match = False
                volume_match = False
                all_match = False
                match_status = "pdf_only"
            else:
                matched_work_ids_for_cell.add(matched.work_id)

                openalex_title = matched.title
                openalex_work_id = matched.work_id
                openalex_doi = matched.doi
                openalex_year = matched.year
                openalex_volume = matched.volume
                openalex_issue = matched.issue
                openalex_source = getattr(matched, "source", pd.NA)
                title_match = score >= 0.86
                year_match = values_match(pdf_year, openalex_year)
                volume_match = values_match(pdf_volume, openalex_volume)
                all_match = bool(title_match and year_match and volume_match)
                if all_match:
                    match_status = "matched"
                elif not title_match:
                    match_status = "matched_with_title_difference"
                else:
                    match_status = "matched_with_metadata_difference"

            validation_rows.append({
                "conference": conference,
                "conference_year": year,
                "pdf_issue": pdf_issue,
                "pdf_year": pdf_year,
                "pdf_volume": pdf_volume,
                "pdf_entry_order": order,
                "pdf_page_or_article": pdf_page_from_block(block),
                "pdf_title": pdf_title,
                "openalex_title": openalex_title,
                "openalex_work_id": openalex_work_id,
                "openalex_doi": openalex_doi,
                "openalex_year": openalex_year,
                "openalex_volume": openalex_volume,
                "openalex_issue": openalex_issue,
                "openalex_source": openalex_source,
                "title_match_score": round(float(score), 3),
                "title_match": title_match,
                "year_match": year_match,
                "volume_match": volume_match,
                "all_match": all_match,
                "match_status": match_status,
                "pdf_path": str(path.relative_to(project_folder)),
            })

    unmatched_data = cell_papers[~cell_papers["work_id"].isin(matched_work_ids_for_cell)]

    for row in unmatched_data.itertuples(index=False):
        validation_rows.append({
            "conference": conference,
            "conference_year": year,
            "pdf_issue": pd.NA,
            "pdf_year": pd.NA,
            "pdf_volume": pd.NA,
            "pdf_entry_order": pd.NA,
            "pdf_page_or_article": pd.NA,
            "pdf_title": pd.NA,
            "openalex_title": row.title,
            "openalex_work_id": row.work_id,
            "openalex_doi": row.doi,
            "openalex_year": row.year,
            "openalex_volume": row.volume,
            "openalex_issue": row.issue,
            "openalex_source": getattr(row, "source", pd.NA),
            "title_match_score": 0.0,
            "title_match": False,
            "year_match": False,
            "volume_match": False,
            "all_match": False,
            "match_status": "openalex_only",
            "pdf_path": pd.NA,
        })

validation_table = pd.DataFrame(validation_rows)
validation_table = validation_table.sort_values(
    ["conference", "conference_year", "pdf_issue", "pdf_entry_order", "openalex_title"],
    na_position="last",
).reset_index(drop=True)

comparison_summary = (
    validation_table
    .groupby(["conference", "conference_year"], dropna=False)
    .agg(
        n_pdf_titles=("pdf_title", lambda values: values.notna().sum()),
        n_openalex_titles=("openalex_title", lambda values: values.notna().sum()),
        n_all_match=("all_match", "sum"),
        n_not_all_match=("all_match", lambda values: (~values).sum()),
        n_pdf_only=("match_status", lambda values: (values == "pdf_only").sum()),
        n_openalex_only=("match_status", lambda values: (values == "openalex_only").sum()),
        n_title_difference=("match_status", lambda values: (values == "matched_with_title_difference").sum()),
        n_metadata_difference=("match_status", lambda values: (values == "matched_with_metadata_difference").sum()),
    )
    .reset_index()
)

display(comparison_summary)

comparison_columns = [
    "conference", "conference_year", "pdf_issue", "pdf_year", "pdf_volume",
    "openalex_year", "openalex_volume", "pdf_title", "openalex_title",
    "openalex_work_id", "title_match", "year_match", "volume_match",
    "all_match", "match_status",
]

display(validation_table[comparison_columns])

problems = validation_table[~validation_table["all_match"]].copy()

print("Rows where something does not match:", len(problems))
if len(problems):
    display(problems[[
        "conference", "conference_year", "pdf_issue", "pdf_title",
        "openalex_title", "openalex_work_id", "pdf_year", "openalex_year",
        "pdf_volume", "openalex_volume", "match_status",
    ]])


## 8 - Save

In [ ]:
per_paper_path = intermediate_dir / "title_check_per_paper.parquet"
summary_path = intermediate_dir / "title_check_summary.parquet"
inventory_path = intermediate_dir / "title_check_pdf_inventory.parquet"
validation_path = intermediate_dir / "paper_title_matching_validation.parquet"
comparison_summary_path = intermediate_dir / "paper_title_matching_summary.parquet"

title_check.to_parquet(per_paper_path, index=False)
summary.to_parquet(summary_path, index=False)
pdf_inventory.to_parquet(inventory_path, index=False)
validation_table.to_parquet(validation_path, index=False)
comparison_summary.to_parquet(comparison_summary_path, index=False)

summary.to_csv(summary_tables_dir / "title_check_summary.csv", index=False)
pdf_inventory.to_csv(check_tables_dir / "title_check_pdf_inventory.csv", index=False)
validation_table.to_csv(check_tables_dir / "paper_title_matching_validation.csv", index=False)
comparison_summary.to_csv(summary_tables_dir / "paper_title_matching_summary.csv", index=False)

print(per_paper_path)
print(summary_path)
print(inventory_path)
print(validation_path)
print(comparison_summary_path)


## Output

| File | Description |
|---|---|
| `data/intermediate/paper_title_checks/title_check_per_paper.parquet` | One row per paper title check |
| `data/intermediate/paper_title_checks/title_check_summary.parquet` | One row per conference year |
| `data/intermediate/paper_title_checks/title_check_pdf_inventory.parquet` | PDFs used for each conference year |
| `data/intermediate/paper_title_checks/paper_title_matching_validation.parquet` | One row per PDF title or unmatched OpenAlex title |
| `data/intermediate/paper_title_checks/paper_title_matching_summary.parquet` | Conference year summary of the matching table |
| `summary_tables/title_check_summary.csv` | CSV summary for report checks |
| `check_tables/paper_title_matching_validation.csv` | Detailed PDF/OpenAlex comparison table |
| `summary_tables/paper_title_matching_summary.csv` | Summary of the detailed comparison table |
